<a href="https://colab.research.google.com/github/arafat17469/Android-APP/blob/main/Sentiment_Analysis_on_IMDB_Movie_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="
  background: linear-gradient(135deg, #0f766e, #14b8a6, #5eead4);
  padding: 38px 28px;
  border-radius: 16px;
  text-align: center;
  color: #ffffff;
  font-family: 'Segoe UI', Arial, sans-serif;
  box-shadow: 0 10px 25px rgba(0,0,0,0.25);
">

  <h1 style="
    margin-bottom: 10px;
    font-size: 32px;
    letter-spacing: 0.5px;
  ">
    🎬 IMDB Sentiment Analysis Project
  </h1>

  <h2 style="
    margin-top: 0;
    font-weight: 400;
    font-size: 20px;
    color: #ecfeff;
  ">
    TF–IDF • Word2Vec • BERT
  </h2>

  <div style="
    width: 70px;
    height: 3px;
    background: #ffffff;
    margin: 16px auto;
    border-radius: 2px;
    opacity: 0.8;
  "></div>

  <p style="
    font-size: 15px;
    margin-top: 10px;
    color: #f0fdfa;
    line-height: 1.6;
    max-width: 680px;
    margin-left: auto;
    margin-right: auto;
  ">
    A comparative study of text representation techniques for sentiment classification,
    focusing on performance, accuracy, and contextual understanding.
  </p>

</div>

<div style="background-color:#fff8e1; padding:16px; border-radius:12px; border-left:6px solid #ffb300;">
  <h2 style="margin-top:0;">🎯 Project Objective</h2>
  <p>
    The goal of this project is to build and compare three different sentiment analysis approaches
    using the official IMDB Movie Reviews dataset.
  </p>
  <ul>
    <li><b>TF–IDF + Logistic Regression</b></li>
    <li><b>Word2Vec + Logistic Regression</b></li>
    <li><b>BERT Embeddings + Classification Head</b></li>
  </ul>
  <p>
    The comparison focuses on:
  </p>
  <ul>
    <li>📊 Classification performance</li>
    <li>⚡ Runtime and computational cost</li>
    <li>🧠 Representation quality</li>
    <li>🔍 Error patterns across models</li>
  </ul>
</div>

<div style="background-color:#e3f2fd; padding:14px; border-radius:12px; border-left:6px solid #1e88e5;">
  <h3 style="margin-top:0;">⚙️ Install Required Libraries</h3>
  <p>
    We install the required libraries for dataset loading, preprocessing, feature extraction,
    machine learning, and transformer-based embeddings.
  </p>
</div>

In [ ]:
!pip install -q datasets transformers gensim scikit-learn nltk tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.8 MB/s eta 0:00:00


<div style="background-color:#e8f5e9; padding:14px; border-radius:12px; border-left:6px solid #43a047;">
  <h3 style="margin-top:0;">📚 Import Required Libraries</h3>
  <p>
    In this step, we import all necessary Python libraries for data handling,
    NLP preprocessing, modeling, and evaluation.
  </p>
</div>

In [ ]:
import re
import html
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import nltk
nltk.download("punkt")

from nltk.tokenize import word_tokenize

from datasets import load_dataset
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

import torch
from transformers import AutoTokenizer, AutoModel

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


<div style="background-color:#f3e5f5; padding:14px; border-radius:12px; border-left:6px solid #8e24aa;">
  <h3 style="margin-top:0;">🔁 Reproducibility Setup</h3>
  <p>
    To ensure consistent and reproducible results, we set fixed random seeds.
  </p>
</div>

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


<div style="background-color:#fff3e0; padding:14px; border-radius:12px; border-left:6px solid #fb8c00;">
  <h3 style="margin-top:0;">📥  Load the Official IMDB Dataset</h3>
  <p>
    We use the official <b>IMDB Movie Reviews</b> dataset from Hugging Face.
    It contains <b>50,000 labeled reviews</b>, balanced between positive and negative sentiments.
  </p>
</div>

In [ ]:
imdb = load_dataset("imdb")
print(imdb)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


<div style="background-color:#ffebee; padding:14px; border-radius:12px; border-left:6px solid #e53935;">
  <h3 style="margin-top:0;">🔀 Create Train, Validation, and Test Sets</h3>
  <p>
    The official IMDB dataset already provides separate training and test sets.
    We further split the official training set into:
  </p>
  <ul>
    <li><b>Training set</b></li>
    <li><b>Validation set</b></li>
  </ul>
  <p>
    The official test set remains untouched for final evaluation, which helps prevent data leakage.
  </p>
</div>

In [ ]:
train_df = pd.DataFrame(imdb["train"])
test_df = pd.DataFrame(imdb["test"])

print("Original training shape:", train_df.shape)
print("Original test shape    :", test_df.shape)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].tolist(),
    train_df["label"].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["label"].tolist()
)

test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

print("Train size     :", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size      :", len(test_texts))

Original training shape: (25000, 2)
Original test shape    : (25000, 2)
Train size     : 20000
Validation size: 5000
Test size      : 25000


<div style="background-color:#e0f7fa; padding:14px; border-radius:12px; border-left:6px solid #00acc1;">
  <h3 style="margin-top:0;">🧹 Define Preprocessing Strategy</h3>
  <p>
    Different models require different preprocessing pipelines.
  </p>
  <h4>For TF–IDF and Word2Vec:</h4>
  <ul>
    <li>Lowercasing</li>
    <li>HTML tag removal</li>
    <li>Punctuation removal</li>
    <li>Whitespace normalization</li>
  </ul>
  <h4>For BERT:</h4>
  <ul>
    <li>Minimal preprocessing</li>
    <li>Lowercasing</li>
    <li>HTML tag removal</li>
    <li>No stopword removal</li>
    <li>No aggressive punctuation removal</li>
  </ul>
</div>

In [ ]:
def remove_html_tags(text):
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    return text

def preprocess_basic(text):
    """
    Used for TF-IDF and Word2Vec
    """
    text = text.lower()
    text = remove_html_tags(text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_bert(text):
    """
    Minimal preprocessing for BERT
    """
    text = text.lower()
    text = remove_html_tags(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return word_tokenize(text)

<div style="background-color:#f1f8e9; padding:14px; border-radius:12px; border-left:6px solid #7cb342;">
  <h3 style="margin-top:0;">🛠️ Apply Preprocessing to All Splits</h3>
  <p>
    We preprocess the training, validation, and test sets consistently.
    Separate preprocessing is maintained for traditional models and BERT.
  </p>
</div>

In [ ]:
train_texts_basic = [preprocess_basic(text) for text in tqdm(train_texts, desc="Preprocessing train (basic)")]
val_texts_basic   = [preprocess_basic(text) for text in tqdm(val_texts, desc="Preprocessing val (basic)")]
test_texts_basic  = [preprocess_basic(text) for text in tqdm(test_texts, desc="Preprocessing test (basic)")]

train_texts_bert = [preprocess_bert(text) for text in tqdm(train_texts, desc="Preprocessing train (BERT)")]
val_texts_bert   = [preprocess_bert(text) for text in tqdm(val_texts, desc="Preprocessing val (BERT)")]
test_texts_bert  = [preprocess_bert(text) for text in tqdm(test_texts, desc="Preprocessing test (BERT)")]

Preprocessing test (BERT): 100%|██████████| 25000/25000 [00:09<00:00, 2662.72it/s]


<div style="background-color:#ede7f6; padding:14px; border-radius:12px; border-left:6px solid #5e35b1;">
  <h3 style="margin-top:0;">📊 Define Evaluation Function</h3>
  <p>
    To compare all models fairly, we use the same evaluation metrics:
  </p>
  <ul>
    <li>Accuracy</li>
    <li>Precision</li>
    <li>Recall</li>
    <li>F1-score</li>
  </ul>
</div>

In [ ]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary"
    )

    print(f"\n{model_name}")
    print("-" * 60)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

<div style="background: linear-gradient(90deg, #283c86, #45a247); padding:16px; border-radius:12px;">
  <h2 style="color:white; margin:0;">🔤 TF–IDF Representation + Logistic Regression</h2>
</div>

<br>

<div style="background-color:#fafafa; padding:14px; border-radius:12px; border-left:6px solid #3949ab;">
  <p>
    TF–IDF converts each review into a sparse numerical vector based on word importance.
    We then train a <b>Logistic Regression</b> classifier on these features.
  </p>
</div>

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts_basic)
X_val_tfidf   = tfidf_vectorizer.transform(val_texts_basic)
X_test_tfidf  = tfidf_vectorizer.transform(test_texts_basic)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF val shape  :", X_val_tfidf.shape)
print("TF-IDF test shape :", X_test_tfidf.shape)

TF-IDF train shape: (20000, 20000)
TF-IDF val shape  : (5000, 20000)
TF-IDF test shape : (25000, 20000)


<div style="background-color:#e8eaf6; padding:14px; border-radius:12px; border-left:6px solid #5c6bc0;">
  <h3 style="margin-top:0;">🤖 Train TF–IDF Classifier</h3>
  <p>
    Logistic Regression is a strong baseline for text classification tasks and works very well with TF–IDF vectors.
  </p>
</div>

In [ ]:
tfidf_clf = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1
)

tfidf_clf.fit(X_train_tfidf, train_labels)

val_preds_tfidf = tfidf_clf.predict(X_val_tfidf)
test_preds_tfidf = tfidf_clf.predict(X_test_tfidf)

<div style="background-color:#fce4ec; padding:14px; border-radius:12px; border-left:6px solid #d81b60;">
  <h3 style="margin-top:0;">✅ Evaluate TF–IDF Model</h3>
</div>

In [ ]:
tfidf_val_results = evaluate_model(val_labels, val_preds_tfidf, model_name="TF-IDF (Validation)")
tfidf_test_results = evaluate_model(test_labels, test_preds_tfidf, model_name="TF-IDF (Test)")


TF-IDF (Validation)
------------------------------------------------------------
Accuracy : 0.8956
Precision: 0.8900
Recall   : 0.9028
F1-score : 0.8963

Classification Report:
              precision    recall  f1-score   support

           0     0.9014    0.8884    0.8948      2500
           1     0.8900    0.9028    0.8963      2500

    accuracy                         0.8956      5000
   macro avg     0.8957    0.8956    0.8956      5000
weighted avg     0.8957    0.8956    0.8956      5000


TF-IDF (Test)
------------------------------------------------------------
Accuracy : 0.8904
Precision: 0.8869
Recall   : 0.8950
F1-score : 0.8909

Classification Report:
              precision    recall  f1-score   support

           0     0.8940    0.8859    0.8899     12500
           1     0.8869    0.8950    0.8909     12500

    accuracy                         0.8904     25000
   macro avg     0.8905    0.8904    0.8904     25000
weighted avg     0.8905    0.8904    0.8904     250

<div style="background: linear-gradient(90deg, #1d4350, #a43931); padding:16px; border-radius:12px;">
  <h2 style="color:white; margin:0;">🤖 Word2Vec Representation + Logistic Regression</h2>
</div>

<br>

<div style="background-color:#fafafa; padding:14px; border-radius:12px; border-left:6px solid #6d4c41;">
  <p>
    Word2Vec learns dense vector representations of words based on context.
    Since Word2Vec produces vectors for individual words, we represent each review
    by averaging all valid word vectors in that review.
  </p>
</div>

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from tqdm import tqdm

# Ensure required tokenizer resources are available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

# Define tokenizer (if not defined earlier)
def tokenize_text(text):
    return word_tokenize(text)

# Tokenization
train_tokens = [tokenize_text(text) for text in tqdm(train_texts_basic, desc="Tokenizing train")]
val_tokens   = [tokenize_text(text) for text in tqdm(val_texts_basic, desc="Tokenizing val")]
test_tokens  = [tokenize_text(text) for text in tqdm(test_texts_basic, desc="Tokenizing test")]

print(train_tokens[0][:20])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
Tokenizing test: 100%|██████████| 25000/25000 [00:17<00:00, 1393.13it/s]

['i', 'have', 'always', 'been', 'a', 'huge', 'james', 'bond', 'fanatic', 'i', 'have', 'seen', 'almost', 'all', 'of', 'the', 'films', 'except', 'for', 'die']


<div style="background-color:#efebe9; padding:14px; border-radius:12px; border-left:6px solid #8d6e63;">
  <h3 style="margin-top:0;">🏗️ Train Word2Vec Embeddings</h3>
  <p>
    The Word2Vec model is trained only on the training set to avoid data leakage.
  </p>
</div>

In [ ]:
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,      # skip-gram
    seed=SEED,
    epochs=10
)

print("Word2Vec vocabulary size:", len(w2v_model.wv))

Word2Vec vocabulary size: 42864


<div style="background-color:#ede7f6; padding:14px; border-radius:12px; border-left:6px solid #7e57c2;">
  <h3 style="margin-top:0;">📦 Convert Reviews into Document Vectors</h3>
  <p>
    Each review is represented by the average of its word embeddings.
  </p>
</div>

In [ ]:
def document_vector(tokens, model, vector_size=100):
    valid_words = [word for word in tokens if word in model.wv]
    if len(valid_words) == 0:
        return np.zeros(vector_size)
    return np.mean(model.wv[valid_words], axis=0)

X_train_w2v = np.array([document_vector(tokens, w2v_model, 100) for tokens in tqdm(train_tokens, desc="Train doc vectors")])
X_val_w2v   = np.array([document_vector(tokens, w2v_model, 100) for tokens in tqdm(val_tokens, desc="Val doc vectors")])
X_test_w2v  = np.array([document_vector(tokens, w2v_model, 100) for tokens in tqdm(test_tokens, desc="Test doc vectors")])

print("Word2Vec train shape:", X_train_w2v.shape)
print("Word2Vec val shape  :", X_val_w2v.shape)
print("Word2Vec test shape :", X_test_w2v.shape)

Test doc vectors: 100%|██████████| 25000/25000 [00:13<00:00, 1791.27it/s]

Word2Vec train shape: (20000, 100)
Word2Vec val shape  : (5000, 100)
Word2Vec test shape : (25000, 100)


<div style="background-color:#e8f5e9; padding:14px; border-radius:12px; border-left:6px solid #2e7d32;">
  <h3 style="margin-top:0;">🤖 Train Classifier on Word2Vec Features</h3>
  <p>
    For fair comparison, we again use Logistic Regression as the classifier.
  </p>
</div>

In [ ]:
w2v_clf = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1
)

w2v_clf.fit(X_train_w2v, train_labels)

val_preds_w2v = w2v_clf.predict(X_val_w2v)
test_preds_w2v = w2v_clf.predict(X_test_w2v)

<div style="background-color:#fff8e1; padding:14px; border-radius:12px; border-left:6px solid #f9a825;">
  <h3 style="margin-top:0;">✅ Evaluate Word2Vec Model</h3>
</div>

In [ ]:
w2v_val_results = evaluate_model(val_labels, val_preds_w2v, model_name="Word2Vec (Validation)")
w2v_test_results = evaluate_model(test_labels, test_preds_w2v, model_name="Word2Vec (Test)")


Word2Vec (Validation)
------------------------------------------------------------
Accuracy : 0.8676
Precision: 0.8653
Recall   : 0.8708
F1-score : 0.8680

Classification Report:
              precision    recall  f1-score   support

           0     0.8700    0.8644    0.8672      2500
           1     0.8653    0.8708    0.8680      2500

    accuracy                         0.8676      5000
   macro avg     0.8676    0.8676    0.8676      5000
weighted avg     0.8676    0.8676    0.8676      5000


Word2Vec (Test)
------------------------------------------------------------
Accuracy : 0.8513
Precision: 0.8582
Recall   : 0.8416
F1-score : 0.8498

Classification Report:
              precision    recall  f1-score   support

           0     0.8446    0.8610    0.8527     12500
           1     0.8582    0.8416    0.8498     12500

    accuracy                         0.8513     25000
   macro avg     0.8514    0.8513    0.8513     25000
weighted avg     0.8514    0.8513    0.8513    

<div style="background: linear-gradient(90deg, #42275a, #734b6d); padding:16px; border-radius:12px;">
  <h2 style="color:white; margin:0;">🧠  BERT Embeddings + Classification Head</h2>
</div>

<br>

<div style="background-color:#fafafa; padding:14px; border-radius:12px; border-left:6px solid #8e24aa;">
  <p>
    For the BERT-based approach, we use a pretrained transformer model to extract contextual embeddings
    and then train a lightweight classification head using Logistic Regression.
  </p>
  <p>
    This satisfies the assignment requirement of using:
    <b>“Fine-tune BERT or use its embeddings with a classification head.”</b>
  </p>
</div>

<div style="background-color:#f3e5f5; padding:14px; border-radius:12px; border-left:6px solid #ab47bc;">
  <h3 style="margin-top:0;">📥 Load Pretrained Tokenizer and Model</h3>
  <p>
    We use <b>DistilBERT</b>, a compact and efficient transformer model suitable for Google Colab.
  </p>
</div>

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

<div style="background-color:#ede7f6; padding:14px; border-radius:12px; border-left:6px solid #673ab7;">
  <h3 style="margin-top:0;">🔍 Define BERT Embedding Extraction Function</h3>
  <p>
    We use the first token representation from the transformer output as the review-level embedding.
  </p>
</div>

In [ ]:
def get_bert_embeddings(texts, tokenizer, model, batch_size=32, max_length=256):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting BERT embeddings"):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_state = outputs.last_hidden_state
            cls_embeddings = last_hidden_state[:, 0, :]

        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

<div style="background-color:#e1f5fe; padding:14px; border-radius:12px; border-left:6px solid #039be5;">
  <h3 style="margin-top:0;">⚡ Extract BERT Embeddings</h3>
  <p>
    We now generate BERT embeddings for the training, validation, and test sets.
    This step may take more time than the TF–IDF and Word2Vec pipelines.
  </p>
</div>

In [ ]:
X_train_bert = get_bert_embeddings(train_texts_bert, tokenizer, bert_model, batch_size=32, max_length=256)
X_val_bert   = get_bert_embeddings(val_texts_bert, tokenizer, bert_model, batch_size=32, max_length=256)
X_test_bert  = get_bert_embeddings(test_texts_bert, tokenizer, bert_model, batch_size=32, max_length=256)

print("BERT train shape:", X_train_bert.shape)
print("BERT val shape  :", X_val_bert.shape)
print("BERT test shape :", X_test_bert.shape)

Extracting BERT embeddings: 100%|██████████| 782/782 [03:33<00:00,  3.66it/s]

BERT train shape: (20000, 768)
BERT val shape  : (5000, 768)
BERT test shape : (25000, 768)


<div style="background-color:#fce4ec; padding:14px; border-radius:12px; border-left:6px solid #ec407a;">
  <h3 style="margin-top:0;">🤖 Train Classification Head on BERT Embeddings</h3>
  <p>
    A Logistic Regression classifier is trained on top of the BERT embeddings.
  </p>
</div>

In [ ]:
bert_clf = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1
)

bert_clf.fit(X_train_bert, train_labels)

val_preds_bert = bert_clf.predict(X_val_bert)
test_preds_bert = bert_clf.predict(X_test_bert)

<div style="background-color:#fff3e0; padding:14px; border-radius:12px; border-left:6px solid #ff7043;">
  <h3 style="margin-top:0;">✅ Evaluate BERT-based Model</h3>
</div>

In [ ]:
bert_val_results = evaluate_model(val_labels, val_preds_bert, model_name="BERT Embeddings (Validation)")
bert_test_results = evaluate_model(test_labels, test_preds_bert, model_name="BERT Embeddings (Test)")


BERT Embeddings (Validation)
------------------------------------------------------------
Accuracy : 0.8684
Precision: 0.8738
Recall   : 0.8612
F1-score : 0.8674

Classification Report:
              precision    recall  f1-score   support

           0     0.8632    0.8756    0.8693      2500
           1     0.8738    0.8612    0.8674      2500

    accuracy                         0.8684      5000
   macro avg     0.8685    0.8684    0.8684      5000
weighted avg     0.8685    0.8684    0.8684      5000


BERT Embeddings (Test)
------------------------------------------------------------
Accuracy : 0.8574
Precision: 0.8629
Recall   : 0.8498
F1-score : 0.8563

Classification Report:
              precision    recall  f1-score   support

           0     0.8520    0.8650    0.8584     12500
           1     0.8629    0.8498    0.8563     12500

    accuracy                         0.8574     25000
   macro avg     0.8574    0.8574    0.8574     25000
weighted avg     0.8574    0.8574

<div style="background: linear-gradient(90deg, #134e5e, #71b280); padding:16px; border-radius:12px;">
  <h2 style="color:white; margin:0;">📈  Compare Final Test Results</h2>
</div>

<br>

<div style="background-color:#fafafa; padding:14px; border-radius:12px; border-left:6px solid #26a69a;">
  <p>
    We combine the final test results of all three models into a single comparison table.
  </p>
</div>

In [ ]:
results_df = pd.DataFrame([
    tfidf_test_results,
    w2v_test_results,
    bert_test_results
])

results_df = results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)
results_df

,Model,Accuracy,Precision,Recall,F1-score
0,TF-IDF (Test),0.89044,0.886942,0.89496,0.890933
1,BERT Embeddings (Test),0.85736,0.862876,0.84976,0.856268
2,Word2Vec (Test),0.85128,0.858215,0.84160,0.849826


<div style="background-color:#e8f5e9; padding:14px; border-radius:12px; border-left:6px solid #43a047;">
  <h3 style="margin-top:0;">🏆 Final Results Table</h3>
</div>

In [ ]:
results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}"
})

,Model,Accuracy,Precision,Recall,F1-score
0,TF-IDF (Test),0.8904,0.8869,0.8950,0.8909
1,BERT Embeddings (Test),0.8574,0.8629,0.8498,0.8563
2,Word2Vec (Test),0.8513,0.8582,0.8416,0.8498


<div style="background: linear-gradient(90deg, #5f2c82, #49a09d); padding:16px; border-radius:12px;">
  <h2 style="color:white; margin:0;">🔍Error Analysis</h2>
</div>

<br>

<div style="background-color:#fafafa; padding:14px; border-radius:12px; border-left:6px solid #7b1fa2;">
  <p>
    To understand model weaknesses, we inspect some misclassified examples.
    This helps reveal common error patterns such as:
  </p>
  <ul>
    <li>Sarcasm</li>
    <li>Mixed sentiment</li>
    <li>Negation</li>
    <li>Long, context-heavy reviews</li>
  </ul>
</div>

In [ ]:
def show_misclassified_examples(texts, y_true, y_pred, model_name="Model", n=5):
    print(f"\nMisclassified Examples: {model_name}")
    print("=" * 90)

    count = 0
    for text, true_label, pred_label in zip(texts, y_true, y_pred):
        if true_label != pred_label:
            print(f"\nTrue Label: {true_label} | Predicted: {pred_label}")
            print(text[:900])
            print("-" * 90)
            count += 1
            if count >= n:
                break

show_misclassified_examples(test_texts, test_labels, test_preds_tfidf, model_name="TF-IDF", n=5)
show_misclassified_examples(test_texts, test_labels, test_preds_w2v, model_name="Word2Vec", n=5)
show_misclassified_examples(test_texts, test_labels, test_preds_bert, model_name="BERT", n=5)


Misclassified Examples: TF-IDF

True Label: 0 | Predicted: 1
First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will not like this movie. Most of these movies may not have the best plots or best actors but I enjoy these kinds of movies for what they are. This movie is much better than any of the movies the other action guys (Segal and Dolph) have thought about putting out the past few years. Van Damme is good in the movie, the movie is only worth watching to Van Damme fans. It is not as good as Wake of Death (which i highly recommend to anyone of likes Van Damme) or In hell but, in my opinion it's worth watching. It has the same type of feel to it as Nowhere to Run. Good fun stuff!
------------------------------------------------------------------------------------------

True Label: 0 | Predicted: 1
Ben, (Rupert Grint), is a deeply unhappy adolescent, the son of his unhappily married parents. His father, (Nicholas Farrell), is a vicar and hi

<div style="background-color:#fffde7; padding:14px; border-radius:12px; border-left:6px solid #fdd835;">
  <h2 style="margin-top:0;">🧠  Discussion and Analysis</h2>
  <p>
    The following bullet points summarize the key findings from the three approaches.
  </p>
</div>

In [ ]:
analysis_points = [
    "In this experiment, TF-IDF achieved the best overall performance, with the highest test accuracy and F1-score among the three approaches.",
    "This result shows that traditional sparse features can still be highly effective for sentiment classification, especially on well-structured benchmark datasets such as IMDB.",
    "Word2Vec performed the weakest overall, likely because averaging word embeddings removes word order and reduces contextual information.",
    "The BERT embedding-based model performed better than Word2Vec, but it did not outperform TF-IDF in this setup.",
    "One likely reason is that BERT was used only as a fixed feature extractor rather than being fine-tuned end-to-end for the sentiment classification task.",
    "TF-IDF was also computationally efficient and much simpler to implement compared with the transformer-based pipeline.",
    "Common error patterns across all models included sarcasm, mixed sentiment, subtle emotional shifts, and long reviews containing both praise and criticism.",
    "Negation and contrastive expressions remained challenging, especially when positive and negative cues appeared in the same review.",
    "These findings suggest that more complex representations do not always guarantee better performance unless they are properly adapted to the task.",
    "Overall, the experiment highlights the importance of comparing both classical and deep learning methods rather than assuming that transformer-based representations will always perform best."
]

for i, point in enumerate(analysis_points, 1):
    print(f"{i}. {point}")

1. In this experiment, TF-IDF achieved the best overall performance, with the highest test accuracy and F1-score among the three approaches.
2. This result shows that traditional sparse features can still be highly effective for sentiment classification, especially on well-structured benchmark datasets such as IMDB.
3. Word2Vec performed the weakest overall, likely because averaging word embeddings removes word order and reduces contextual information.
4. The BERT embedding-based model performed better than Word2Vec, but it did not outperform TF-IDF in this setup.
5. One likely reason is that BERT was used only as a fixed feature extractor rather than being fine-tuned end-to-end for the sentiment classification task.
6. TF-IDF was also computationally efficient and much simpler to implement compared with the transformer-based pipeline.
7. Common error patterns across all models included sarcasm, mixed sentiment, subtle emotional shifts, and long reviews containing both praise and criti

<div style="background: linear-gradient(90deg, #11998e, #38ef7d); padding:16px; border-radius:12px;">
  <h2 style="color:white; margin:0;">📝 Final Conclusion</h2>
</div>

<br>

<div style="background-color:#fafafa; padding:14px; border-radius:12px; border-left:6px solid #2e7d32;">
  <p>
    In this final section, we summarize the overall findings of the project in approximately 250 words.
  </p>
</div>

In [ ]:
final_conclusion = """
In this project, three sentiment analysis approaches were implemented and compared on the IMDB Movie Reviews dataset: TF-IDF, Word2Vec, and BERT-based embeddings. The results showed clear differences in performance across the three text representation methods. Among them, TF-IDF achieved the best overall results, obtaining the highest test accuracy and F1-score. This indicates that traditional sparse representations can still be highly effective for sentiment classification, particularly on benchmark datasets with clear lexical sentiment patterns.

The Word2Vec approach produced the weakest performance among the three models. Although Word2Vec captures semantic similarity at the word level, representing each review by averaging word vectors caused loss of word order and contextual structure. The BERT-based embedding model performed better than Word2Vec, but it did not outperform TF-IDF in this experiment. A likely reason is that BERT was used only as a fixed feature extractor with a separate classification head, rather than being fine-tuned directly on the IMDB sentiment task.

Error analysis revealed that all three models struggled with sarcasm, mixed sentiment, subtle emotional tone, and long reviews containing both positive and negative opinions. These challenging cases show that sentiment analysis requires deeper contextual understanding beyond simple keyword matching.

Overall, this project demonstrates that model effectiveness depends not only on the complexity of the representation but also on how well it is adapted to the task. In this experiment, TF-IDF provided the best balance of performance, simplicity, and computational efficiency, while BERT showed promise but would likely benefit from task-specific fine-tuning.
"""
print(final_conclusion)


In this project, three sentiment analysis approaches were implemented and compared on the IMDB Movie Reviews dataset: TF-IDF, Word2Vec, and BERT-based embeddings. The results showed clear differences in performance across the three text representation methods. Among them, TF-IDF achieved the best overall results, obtaining the highest test accuracy and F1-score. This indicates that traditional sparse representations can still be highly effective for sentiment classification, particularly on benchmark datasets with clear lexical sentiment patterns.

The Word2Vec approach produced the weakest performance among the three models. Although Word2Vec captures semantic similarity at the word level, representing each review by averaging word vectors caused loss of word order and contextual structure. The BERT-based embedding model performed better than Word2Vec, but it did not outperform TF-IDF in this experiment. A likely reason is that BERT was used only as a fixed feature extractor with a s

<div style="background-color:#e1f5fe; padding:14px; border-radius:12px; border-left:6px solid #0288d1;">
  <h3 style="margin-top:0;">💾 Save Final Results</h3>
  <p>
    We save the comparison table as a CSV file for reproducibility and future reference.
  </p>
</div>

In [ ]:
results_df.to_csv("imdb_sentiment_comparison_results.csv", index=False)
print("Saved file: imdb_sentiment_comparison_results.csv")

Saved file: imdb_sentiment_comparison_results.csv


<div style="background: linear-gradient(90deg, #232526, #414345); padding: 20px; border-radius: 14px;">
  <h2 style="color: white; text-align: center; margin: 0;">✅ Project Completed Successfully</h2>
  <p style="color: #f1f1f1; text-align: center; margin-top: 10px;">
    TF–IDF, Word2Vec, and BERT-based models were implemented, evaluated, and compared successfully.
  </p>
</div>

In [ ]:
print("Assignment completed successfully.\n")
print("Final Test Results:")
display(results_df)

Assignment completed successfully.

Final Test Results:


,Model,Accuracy,Precision,Recall,F1-score
0,TF-IDF (Test),0.89044,0.886942,0.89496,0.890933
1,BERT Embeddings (Test),0.85736,0.862876,0.84976,0.856268
2,Word2Vec (Test),0.85128,0.858215,0.84160,0.849826
